In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default='browser'
import pyarrow.feather as feather
import plotly.express as px
import numpy as np
import seaborn as sns
import yaml


In [ ]:
#load model parameters from yaml file
with open('model_pars.yaml', 'r') as file:
    model_params = yaml.safe_load(file)

#Access parameters


mineral = model_params['response']
covariates = model_params['covariates']


#model_params = model_params['model']

learning_rate = model_params['learning_rate']
batch_size = model_params['batch_size']
epochs = model_params['epochs']
cellsize = model_params['cellsize']
layers = model_params['layers']
dropout = model_params['dropout']
activation = model_params['activation']
loss = model_params['loss']
optimizer = model_params['optimizer']

x = model_params['x']
y = model_params['y']
z = model_params['z']

# plotting parameters
cov_labels = model_params['cov_labels']

# file paths
source_dir = model_params['source_dir']
working_dir = model_params['working_dir']
roi_dir = model_params['roi_dir']

In [ ]:
model_data = pd.read_csv(f'{source_dir}/Gonneville_full.csv', low_memory=False)  # Gonneville data
# load ROI coords
ROI = pd.read_csv(f'{roi_dir}/ROI.csv', low_memory=False)


In [ ]:
# Assume ROI is a DataFrame with columns: Xmin, Xmax, Ymin, Ymax
# And gonneville_data has columns: mid_x_y, mid_y_y, mid_z_y

#small_ROI = 0
#medium_ROI = 1
#large_ROI = 2

roi_row = 0  # Change this to select the desired ROI row


xmin, xmax = ROI['Xmin'].iloc[roi_row], ROI['Xmax'].iloc[roi_row]
ymin, ymax = ROI['Ymin'].iloc[roi_row], ROI['Ymax'].iloc[roi_row]

filtered_data = model_data[
    (model_data['mid_x_y'] >= xmin) & (model_data['mid_x_y'] <= xmax) &
    (model_data['mid_y_y'] >= ymin) & (model_data['mid_y_y'] <= ymax)
]

In [ ]:
#df = density_data.copy()
df = filtered_data.copy()

#filtered_samples = df.dropna(subset=['Alt1_Code', 'Alt2_Code', 'Density_gcm3'])
#sample_list = Gonneville_pars(response+covariates) # TODO fix this with Yaml
filtered_samples = df.dropna(subset=covariates)
cols = [mineral] + covariates
combined_data = df[cols].dropna()
#deposit_data = deposit_data.dropna

sample_count = len(filtered_samples)
perc = (sample_count/len(df))*100 

print(f"Number of samples with response variable and covariates: {sample_count}")
#print(f"This is " {perc} " percent of the dataset") 
print(perc)

In [ ]:
# Calculate the percentage of the volume that has been sampled by drilling

# Volume of a cylinder = π * r^2 * h
# Assuming a drill core diameter of 10 cm and an interval length of 1 m

# core diameters: AQ: 0.027 m; BQ: 0.0365 m; NQ: 0.0476 m; HQ: 0.0635 m; PQ: 0.085 m 

drill_core_diameter = 0.0635  # in meters
drill_core_radius = drill_core_diameter / 2
drill_core_interval_length = 1 # in meters
# multiply by the number of samples to get the total volume sampled
drill_core_volume = np.pi * (drill_core_radius ** 2) * drill_core_interval_length * len(combined_data) # in cubic meters

# Volume of the domain

domain_volume = (xmax - xmin) * (ymax - ymin) * (df['mid_z_y'].max() - df['mid_z_y'].min())  # in cubic meters
sampled_volume = sample_count * drill_core_volume  # in cubic meters
sampled_volume_percentage = (sampled_volume / domain_volume) * 100
print(f"Sampled volume percentage: {sampled_volume_percentage:.2f}%")



In [ ]:
# Plot missing values in merged_df as two separate heatmaps: 
# 1. For the first half of columns, 2. For the second half

import matplotlib.pyplot as plt
import seaborn as sns

n_cols = combined_data.shape[1]
mid = n_cols // 2

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

# First half of columns
sns.heatmap(
    combined_data.iloc[:, :mid].isnull(),
    cbar=False, yticklabels=False, cmap='viridis', ax=axes[0]
)
axes[0].set_title('Missing Values (First Half of Columns)')
axes[0].set_xlabel('Columns')
axes[0].set_ylabel('Rows')

# Second half of columns
sns.heatmap(
    combined_data.iloc[:, mid:].isnull(),
    cbar=False, yticklabels=False, cmap='viridis', ax=axes[1]
)
axes[1].set_title('Missing Values (Second Half of Columns)')
axes[1].set_xlabel('Columns')

plt.tight_layout()
plt.show()

In [ ]:
# log transform the data


plot_data = combined_data.copy()
plot_data['Log_Pd_ppm'] = np.log(combined_data['Pd_ppm']+0.00001)
cols_to_add = ['mid_x_y', 'mid_y_y', 'mid_z_y']
plot_data = pd.concat([plot_data, filtered_data[cols_to_add]], axis=1)

#TODO get all this from Yaml
# Create 3D scatter plot
fig = px.scatter_3d(plot_data, \
                    x='mid_x_y', y='mid_y_y', z='mid_z_y', \
                    #x='x', y='y', z='z', \ #YAML
                        color='Log_Pd_ppm', \
                        opacity=0.7, labels={'Log_Pd_ppm': 'Pd log(ppm)'})

fig.update_traces(marker=dict(line=dict(width=0)))
# Show plot
fig.show()

In [ ]:
'''two_composite_1covariates = deposit_data

missing_percentage = (two_composite_1covariates.isnull().sum() / len(two_composite_1covariates)) * 100

variables_to_remove = missing_percentage[missing_percentage > 5].index

two_composite_1covariates = two_composite_1covariates.drop(columns=variables_to_remove)
'''
# Find columns with object or category dtype
categorical_cols = combined_data.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_data = combined_data[categorical_cols]

encoded_data = categorical_data.copy()

threshold = 10

for column in categorical_data.columns:
    if categorical_data[column].dtype == 'object':
        unique_values = categorical_data[column].nunique()
        
        if unique_values <= threshold:
            encoded_columns = pd.get_dummies(encoded_data[column], prefix=column)
            encoded_columns = encoded_columns.astype(int)  
            encoded_data = pd.concat([encoded_data, encoded_columns], axis=1)
            encoded_data = encoded_data.drop(columns=[column])




encoded_columns = [col for col in encoded_data.columns if '_Code_' in col]

total_columns = []
'''
for col in encoded_columns:
    prefix, original_column = col.split('_Code_')
    
    total_column = f"{original_column}_Total"
    total_columns.append(total_column)
    
    encoded_data[total_column] = (
        encoded_data[f"{prefix}_Code_{original_column}"] * encoded_data[f"{prefix}_Pct"]
    )

total_columns = []

category_totals = {}

for col in encoded_columns:
    prefix, original_column = col.split('_Code_')
    
    total_column = f"{original_column}_Total"
    if total_column not in total_columns:
        total_columns.append(total_column)
    
    total_values = encoded_data[f"{prefix}_Code_{original_column}"] * encoded_data[f"{prefix}_Pct"]
    
    if total_column in category_totals:
        category_totals[total_column] += total_values
    else:
        category_totals[total_column] = total_values

for total_column, total_values in category_totals.items():
    encoded_data[total_column] = total_values
'''

## SPLOM of numeric variables

In [ ]:

# candidate variables (response + covariates)
candidate_vars = [mineral] + covariates
vars_to_plot = [v for v in candidate_vars if v in combined_data.columns and pd.api.types.is_numeric_dtype(combined_data[v])]

# choose which variables to display on a log axis (example)
#log_vars = ['Pd_ppm', 'Au_ppm']  # replace with your variable names
log_vars = [mineral] + covariates  # replace with your variable names

# drop rows with NA for selected plotting vars
df_plot = combined_data[vars_to_plot].dropna()

# remove non-positive values for log_vars
for lv in log_vars:
    if lv in df_plot.columns:
        df_plot = df_plot[df_plot[lv] > 0]

# optional subsample
#if len(df_plot) > 2000:
#    df_plot = df_plot.sample(2000, random_state=42)

g = sns.pairplot(df_plot, diag_kind="kde", plot_kws={"s": 12, "alpha": 0.6})
#g = sns.PairGrid(df_plot)
#g.map_upper(sns.scatterplot)
#g.map_lower(sns.kdeplot)
#g.map_diag(sns.kdeplot, lw=3, legend=False)

vars_list = df_plot.columns.tolist()
for i, yvar in enumerate(vars_list):
    for j, xvar in enumerate(vars_list):
        ax = g.axes[i, j]
        if xvar in log_vars:
            ax.set_xscale('log')
        if yvar in log_vars:
            ax.set_yscale('log')

g.figure.suptitle("Scatterplot matrix — numeric variables only", y=1.02)
plt.show()



## Plot categorical labels and counts as a histogram

In [ ]:


# get categorical columns
categorical_cols = combined_data.select_dtypes(include=['object','category']).columns.tolist()

for col in categorical_cols:
    counts = combined_data[col].fillna('NaN').value_counts()
    print(f"\nColumn: {col}\n", counts)

    plt.figure(figsize=(8, max(3, len(counts) * 0.35)))
    sns.barplot(x=counts.values, y=counts.index, palette='tab10')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.title(f'Counts per label — {col}')

    # add count labels on bars
    for i, v in enumerate(counts.values):
        plt.text(v, i, f' {v}', va='center')

    plt.tight_layout()
    plt.show()

## Export un-normalised data

In [ ]:
# export unnormalised deposit data
combined_data = pd.concat([combined_data, filtered_data[cols_to_add]], axis=1)


combined_data.to_csv(f'{working_dir}/filtered_unnormalised_deposit_data.csv', index=False)




## Scale response variable and covariates

In [ ]:
combined_data_norm = combined_data.copy()

exclude_cols = [x, y, z] # exclude x, y, z from scaling as they are coordinates and have been normalized
numeric_cols = combined_data_norm.select_dtypes(include='number').columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in exclude_cols]

print(numeric_cols)

scaler = MinMaxScaler()
combined_data_norm.loc[:, numeric_cols] = scaler.fit_transform(combined_data_norm.loc[:,numeric_cols])


## Export data

In [ ]:
all_indices = combined_data_norm.index

all_indices.to_series().to_csv(f'{working_dir}/indices.csv', index=False)
combined_data_norm.to_csv(f'{working_dir}/combined_data_norm.csv', index=False)